# CCN Tutorial · Notebook 2
## Simulate → fit → **recover**: parameter estimation and the (non-)identifiability of linear dynamical systems

In Notebook 1 the parameters $\theta=(A,C,Q,R, m_0, V_0)$ were **known**, and we did inference over states. Here we do the opposite: we observe only $y_{1:T}$ and must **learn $\theta$**. The workhorse is Expectation–Maximization (EM), and the two halves you saw at the board map onto objects from Notebook 1:

- **E-step** = run a Kalman **smoother** with the current $\theta$ to get the posterior over latents (the non-causal sibling of the filter, using *all* of $y_{1:T}$);
- **M-step** = update $\theta$ in closed form from those posterior expectations.

We won't re-derive the M-step — we call `dynamax`'s `fit_em` (on its conjugate LG-SSM, i.e. MAP-EM under weak priors, which keeps every M-step well-conditioned). The scientific point of this notebook is what comes *after* fitting: **a state-space model is only identifiable up to a change of latent basis**, so "did we recover the truth?" is a subtler question than it looks. We'll see what is recoverable (eigenvalues, the Hankel spectrum, the latent trajectory *after alignment*, the model order — which we also pick by cross-validated held-out log-likelihood — and, once each model is put into a canonical *balanced* basis, every parameter matrix) and what is not (the parameters in the basis EM happens to return them in, the latent coordinate system itself).

The ground-truth system is deliberately unfriendly: non-normal dynamics that amplify transiently, heteroskedastic process noise, and spatially correlated observation noise.

> **Notation.** We use the dynamical-systems convention: $A$ = dynamics matrix, $C$ = emission/loading matrix. These are the same objects `dynamax` stores as `dynamics.weights` and `emissions.weights`, and that Notebook 1 wrote as $F$ and $H$.

> **In R?** `02_lds_parameter_recovery_R.ipynb` is the same notebook built on [MARSS](https://atsa-es.github.io/MARSS/) instead of `dynamax`.


## 0. Setup

In [ ]:
try:
    import dynamax  # noqa: F401
except ImportError:
    import subprocess, sys

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "dynamax"], check=True
    )

import numpy as np, jax

jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import vmap, random
from scipy.linalg import solve_discrete_lyapunov, solve_triangular
from dynamax.linear_gaussian_ssm import LinearGaussianSSM, LinearGaussianConjugateSSM
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
plt.rcParams.update(
    {
        "figure.dpi": 300,
        "font.size": 11,
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.alpha": 0.25,
    }
)
rng = np.random.default_rng(1)

## 1. The model, and the symmetry that haunts it

The linear–Gaussian SSM, as in Notebook 1:
$$
x_t = A\,x_{t-1} + w_t,\quad w_t\sim\mathcal N(0,Q);\qquad
y_t = C\,x_t + v_t,\quad v_t\sim\mathcal N(0,R);\qquad x_0\sim\mathcal N( m_0, V_0).
$$

Here is the fact that shapes everything below. Pick **any** invertible matrix $T\in\mathrm{GL}(d_x)$ and relabel the latent state $x_t \mapsto T x_t$. Absorbing $T$ into the parameters,
$$
A \mapsto T A T^{-1},\quad
C \mapsto C T^{-1},\quad
Q \mapsto T Q T^{\top},\quad
 m_0 \mapsto T m_0,\quad
 V_0 \mapsto T V_0 T^{\top},\quad
R \mapsto R,
$$
gives a **different** parameter set that induces the **identical** distribution over the observations $p(y_{1:T})$. The likelihood is exactly flat along this $d_x^2$-dimensional group orbit, so no amount of data can pin down *which* member of the orbit generated the data.

**Consequences.**
- Comparing a fitted $\hat A$ to the true $A$ entry-by-entry is meaningless — they can differ wildly yet describe the same system.
- Comparing fitted latent trajectories to true ones requires first **undoing** the basis change (alignment).
- What *is* estimable are quantities **invariant** to $T$. Two we will use: the **eigenvalues of $A$** (a similarity transform preserves them) and the **Hankel matrix of output covariances** (a property of $y$ itself).


## 2. A ground-truth system

We build a $d_x=5$ latent with a deliberately readable spectrum — two **complex-conjugate pairs** (damped rotations of period $\approx20$ and $\approx8$ steps, with $|\lambda|=0.95$ and $0.85$) plus one leftover **real** mode ($\lambda=0.9$, slow decay) — then rotate the coordinates by a random orthogonal matrix so the modes are mixed across all latent dimensions (nothing is secretly axis-aligned). It is read out into $d_y=50$ noisy channels.

Three further ingredients make the problem harder, and more realistic, than a toy:

- **Non-normal dynamics.** Feedforward coupling is placed strictly *above* the modal block-diagonal, so $A$ stays block-upper-triangular: the eigenvalues — and hence stability — are exactly unchanged, but the eigenvectors become oblique and the system **amplifies transiently** before decaying (functionally feedforward / balanced amplification). Set `nonnormal = False` for the normal, orthogonal-mode case.
- **Heteroskedastic process noise.** $Q$ is diagonal but far from a multiple of the identity, so no latent direction is privileged by construction.
- **Structured observation noise.** Channel variances differ (one is deliberately terrible), *and* channels are spatially autocorrelated: $R = D^{1/2}KD^{1/2}$ with $K_{ij}=\exp(-|i-j|/\ell)$. Correlated noise concentrates into a few large eigenvalues — which is exactly how it can masquerade as extra latent dimensions. The printed **effective rank** of $R$ says how many channels' worth of noise you are really up against.

Finally, the loadings $C$ are rescaled to hit a fixed per-channel **signal-to-noise ratio**. Without that, latent power swings by orders of magnitude as $d_x$, the feedforward gain, or $\ell$ change (non-normal amplification alone moves it $\sim1000\times$), so nothing would be comparable across settings.

In [ ]:
dx, dy = 5, 50

# ---------------------------------------------------------------- latent dynamics
# Rotational modes occupy coordinate *pairs*: (1,2), (3,4), ... With dx odd the leftover
# coordinate is a single real mode. Non-normality is feedforward coupling placed strictly
# ABOVE the modal block-diagonal, which keeps A block-upper-triangular: the eigenvalues --
# and hence stability -- are exactly unchanged, while the eigenvectors become oblique and
# the system amplifies transiently before decaying (functionally feedforward /
# balanced amplification). Set nonnormal = False for the normal (orthogonal-mode) case.
nonnormal = True
ff_gain = 1.0
ff_mode = (
    "chain"  # if nonnormal: "last": only the final block drives the others (gentle);
)
# "chain": each block drives the previous one, so amplification compounds with dx.
real_eig = 0.9  # eigenvalue of the leftover real mode (odd dx only)
periods = (20, 8)  # rotation period of the slowest / fastest pair
radii = (0.95, 0.85)  # |lambda| of the slowest / fastest pair

n_pair = dx // 2
per, rad = np.linspace(*periods, n_pair), np.linspace(*radii, n_pair)
A_modal = np.zeros((dx, dx))
blocks = []  # (start, stop) of each diagonal block, in order
for k in range(n_pair):
    th, s = 2 * np.pi / per[k], 2 * k
    A_modal[s : s + 2, s : s + 2] = rad[k] * np.array(
        [[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]]
    )
    blocks.append((s, s + 2))
if dx % 2:
    A_modal[-1, -1] = real_eig
    blocks.append((dx - 1, dx))

if nonnormal and len(blocks) > 1:  # a single block cannot be made non-normal
    if ff_mode == "chain":
        for (a0, a1), (b0, b1) in zip(blocks[:-1], blocks[1:]):
            A_modal[a0:a1, b0:b1] = ff_gain  # block k+1 drives block k
    else:
        lo = blocks[-1][0]
        A_modal[:lo, lo:] = ff_gain  # last block drives everything upstream

M = np.linalg.qr(rng.normal(size=(dx, dx)))[0]  # random orthogonal mixing
A = M @ A_modal @ M.T

Q = 0.1 * np.eye(dx) + 0.9 * np.diag(rng.normal(size=dx) ** 2)  # heteroskedastic noise
C = rng.normal(size=(dy, dx))
C /= np.linalg.norm(C, axis=0, keepdims=True)
snr = 0.5  # per-channel signal-to-noise ratio; None keeps C exactly as drawn

# ------------------------------------------------------------- observation noise
# Channels differ in variance AND are spatially autocorrelated: R = D^(1/2) K D^(1/2)
# with K_ij = exp(-d_ij / obs_ell). That Toeplitz K is positive definite for any
# obs_ell > 0, and its inverse is tridiagonal -- i.e. this is exactly the Gaussian
# Markov random field whose precision is a chain-graph Laplacian, so "Toeplitz" and
# "graph Laplacian" are the same object here. obs_ell = 0 gives uncorrelated channels.
obs_ell = 5.0  # spatial correlation length, in channels (0 = white)
obs_ring = False  # True: channels sit on a ring, so distance wraps around

chan_var = 0.1 + 0.9 * rng.normal(size=dy) ** 2  # heteroskedastic channel variances
chan_var[0] += np.sqrt(chan_var.sum())  # one deliberately terrible channel
d_ij = np.abs(np.subtract.outer(np.arange(dy), np.arange(dy)))
if obs_ring:
    d_ij = np.minimum(d_ij, dy - d_ij)
K_corr = np.exp(-d_ij / obs_ell) if obs_ell > 0 else np.eye(dy)
R = K_corr * np.sqrt(np.outer(chan_var, chan_var))

# Fix the signal-to-noise ratio explicitly. Without this, the latent power swings by
# orders of magnitude as dx / ff_gain / obs_ell change (non-normal amplification alone
# moves it ~1000x), so nothing is comparable across settings.
Pi_true = solve_discrete_lyapunov(A, Q)
if snr is not None:
    C *= np.sqrt(snr * np.mean(np.diag(R)) / np.mean(np.diag(C @ Pi_true @ C.T)))

m_0, V_0 = rng.normal(size=dx), 2 * np.eye(dx)

eig_true = np.sort_complex(np.linalg.eigvals(A))
print("true eigenvalues of A:", eig_true)
print("|lambda| =", np.abs(eig_true))

# Stability is set by the spectrum; transient amplification is set by non-normality.
gain = [np.linalg.norm(np.linalg.matrix_power(A, t), 2) for t in range(1, 81)]
print(
    f"spectral radius = {np.abs(eig_true).max():.3f}  (stable: {np.abs(eig_true).max() < 1})"
)
print(f"non-normality ||A'A - AA'||_F = {np.linalg.norm(A.T @ A - A @ A.T):.2f}")
print(
    f"peak transient gain max_t ||A^t||_2 = {max(gain):.2f} at t = {int(np.argmax(gain)) + 1}"
)

# How separable are signal and noise? Correlated noise concentrates into a few large
# eigenvalues, which is exactly how it can masquerade as extra latent dimensions.
sig_ev = np.linalg.eigvalsh(C @ Pi_true @ C.T)[::-1][:dx]
noise_ev = np.linalg.eigvalsh(R)[::-1]
eff_rank = noise_ev.sum() ** 2 / (noise_ev**2).sum()
print(f"\nsignal eigenvalues (C Pi C') = {np.round(sig_ev, 2)}")
print(f"top noise eigenvalues of R   = {np.round(noise_ev[:3], 2)}")
print(f"effective rank of R = {eff_rank:.1f} of {dy} channels")


def build_true_model(A, Q, C, R, m_0, V_0):
    m = LinearGaussianSSM(A.shape[0], C.shape[0])
    p, _ = m.initialize(
        random.PRNGKey(0),
        initial_mean=jnp.array(m_0),
        initial_covariance=jnp.array(V_0),
        dynamics_weights=jnp.array(A),
        dynamics_covariance=jnp.array(Q),
        emission_weights=jnp.array(C),
        emission_covariance=jnp.array(R),
    )
    return m, p


m_true, p_true = build_true_model(A, Q, C, R, m_0, V_0)

In [ ]:
figsize = (15, 9.5)


# Figure 1a: the parameters themselves, as heat maps.
def show_mat(
    ax, M, title, xticklabels=None, yticklabels=None, annot=False, aspect="equal"
):
    # One parameter matrix, on its own color scale (magnitudes differ several-fold, so a
    # shared scale would flatten Q to nothing). Signed matrices get a diverging map
    # centered at zero; non-negative ones a sequential map anchored at zero.
    M = np.atleast_2d(M)
    if M.min() < 0:
        v = np.abs(M).max()
        im = ax.imshow(M, cmap="RdBu_r", vmin=-v, vmax=v, aspect=aspect)
    else:
        im = ax.imshow(
            M,
            cmap="viridis",
            vmin=0,
            vmax=M.max() if M.max() > 0 else 1.0,
            aspect=aspect,
        )
    ax.set_title(title, fontsize=10, pad=6)
    ax.grid(False)
    for lbls, axis, n in (
        (xticklabels, "x", M.shape[1]),
        (yticklabels, "y", M.shape[0]),
    ):
        set_ticks, set_lbls = (
            (ax.set_xticks, ax.set_xticklabels)
            if axis == "x"
            else (ax.set_yticks, ax.set_yticklabels)
        )
        if lbls is None:  # plain 1-based indices, thinned when there are many
            ticks = np.arange(n)[:: 1 if n <= 6 else int(np.ceil(n / 6))]
            set_ticks(ticks)
            set_lbls([str(i + 1) for i in ticks], fontsize=8)
        else:
            set_ticks(np.arange(len(lbls)))
            set_lbls(lbls, fontsize=9)
    if annot:
        for i in range(M.shape[0]):
            for j in range(M.shape[1]):
                r, g, b, _ = im.cmap(im.norm(M[i, j]))  # keep text readable on any cell
                col = "k" if 0.30 * r + 0.59 * g + 0.11 * b > 0.55 else "w"
                ax.text(
                    j,
                    i,
                    f"{M[i, j]:.2f}",
                    ha="center",
                    va="center",
                    color=col,
                    fontsize=9,
                )
    cb = ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cb.ax.tick_params(labelsize=8)
    return im


lat = [f"$x_{i + 1}$" for i in range(dx)]  # one label per latent dimension
fig = plt.figure(figsize=figsize)
outer = fig.add_gridspec(2, 1, height_ratios=[1, 1.7], hspace=0.45)
top = outer[0].subgridspec(1, 4, wspace=0.75)
bot = outer[1].subgridspec(1, 2, width_ratios=[1, 2.6], wspace=0.05)
show_mat(
    fig.add_subplot(top[0]),
    m_0[:, None],
    r"$m_0$  (prior mean)",
    [""],
    lat,
    annot=True,
    aspect="auto",
)
show_mat(
    fig.add_subplot(top[1]),
    V_0,
    r"$V_0$  (prior cov.)",
    lat,
    lat,
    annot=True,
    aspect="auto",
)
show_mat(
    fig.add_subplot(top[2]), A, r"$A$  (dynamics)", lat, lat, annot=True, aspect="auto"
)
show_mat(
    fig.add_subplot(top[3]),
    Q,
    r"$Q$  (process noise)",
    lat,
    lat,
    annot=True,
    aspect="auto",
)
ax_C = fig.add_subplot(bot[0])
show_mat(ax_C, C, rf"$C$  (loadings, ${dy}\times{dx}$)", lat, None)
ax_C.tick_params(axis="x", labelrotation=90)  # the panel is narrow when dy >> dx
show_mat(fig.add_subplot(bot[1]), R, rf"$R$  (obs. noise, ${dy}\times{dy}$)")
fig.suptitle(
    r"Model parameters $\theta=(A,Q,C,R,m_0,V_0)$ — each panel on its own color scale",
    fontsize=12,
)
plt.show()

# Figure 1b: eigenvalues of A, and the latent phase portrait.
x_demo = np.array(m_true.sample(p_true, random.PRNGKey(0), 300)[0])  # one trajectory

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
phi = np.linspace(0, 2 * np.pi, 200)
ax[0].plot(np.cos(phi), np.sin(phi), "k--", lw=1)
ax[0].scatter(eig_true.real, eig_true.imag, s=80, zorder=3, color="C3")
ax[0].axhline(0, color="0.7", lw=0.6)
ax[0].axvline(0, color="0.7", lw=0.6)
ax[0].set(
    title="Eigenvalues of $A$ (unit circle)",
    xlabel="Re",
    ylabel="Im",
    xlim=(-1.15, 1.15),
    ylim=(-1.15, 1.15),
    aspect="equal",
)

ax[1].plot(x_demo[:, 0], x_demo[:, 1], lw=1, color="C0")
ax[1].scatter(*x_demo[0, :2], color="C2", zorder=3, label="start")
ax[1].scatter(*x_demo[-1, :2], color="C3", zorder=3, label="end")
ax[1].set(
    title=f"Latent trajectory ({dx}-D, projected on $x_1$–$x_2$)",
    xlabel="$x_1$",
    ylabel="$x_2$",
)
ax[1].legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# Simulate training and validation batches of independent sequences.
n_train, n_val, T = 50, 25, 250


def sample_batch(model, params, n, T, seed):
    keys = random.split(random.PRNGKey(seed), n)
    states, emis = vmap(lambda k: model.sample(params, k, T))(keys)
    return np.array(states), np.array(emis)


X_train, Y_train = sample_batch(m_true, p_true, n_train, T, 10)
X_val, Y_val = sample_batch(m_true, p_true, n_val, T, 999)
print("Y_train:", Y_train.shape, " Y_val:", Y_val.shape)

## 3. The latent basis is genuinely unidentifiable — a demonstration

Before fitting anything, let's *see* the symmetry. Take the true parameters, apply a random invertible $T$, and check two things on held-out data: (i) the marginal log-likelihood is unchanged, and (ii) the smoothed latents of the two models are related by exactly $T$.

In [ ]:
T_mat = rng.normal(size=(dx, dx))
T_inv = np.linalg.inv(T_mat)
m_tf, p_tf = build_true_model(
    T_mat @ A @ T_inv,
    T_mat @ Q @ T_mat.T,
    C @ T_inv,
    R,
    T_mat @ m_0,
    T_mat @ V_0 @ T_mat.T,
)

mll_true = float(
    vmap(lambda y: m_true.marginal_log_prob(p_true, y))(jnp.array(Y_val)).sum()
)
mll_tf = float(vmap(lambda y: m_tf.marginal_log_prob(p_tf, y))(jnp.array(Y_val)).sum())
print(f"marginal log-lik (val):   true model = {mll_true:.4f}")
print(f"                     T-transformed = {mll_tf:.4f}")
print(f"                          difference = {abs(mll_true - mll_tf):.2e}")

s0 = vmap(lambda y: m_true.smoother(p_true, y))(jnp.array(Y_val)).smoothed_means
s1 = vmap(lambda y: m_tf.smoother(p_tf, y))(jnp.array(Y_val)).smoothed_means
print(
    f"\nmax | smoothed(T-model)  -  T . smoothed(true) | = "
    f"{np.abs(np.array(s1) - np.array(s0) @ T_mat.T).max():.2e}"
)

Identical likelihood; latents that map onto each other by $T$. The latent coordinate system is a *modeling choice*, not something the data determine. So from here on we compare **invariants**, or we **align** before comparing.

## 4. Fit a fresh model with EM

We now discard the truth and fit $\theta$ from $Y_\text{train}$ alone. Three practical notes:
- We use `dynamax`'s **conjugate** LG-SSM (`LinearGaussianConjugateSSM`), i.e. MAP-EM under weak matrix-normal-inverse-Wishart priors. Plain MLE-EM for LDS is prone to a covariance losing positive-definiteness and diverging; the mild prior keeps every M-step well-conditioned at negligible bias.
- We initialize the emission matrix $C$ from the top principal components of the data — a cheap, standard warm start.
- We also keep the **pre-EM initialization** itself (`p_init`, rebuilt with the same seed as the fit). Sections 7 and 8 score it against the truth alongside the fitted model, which is what turns "how far did EM travel?" into a number rather than a rhetorical question.

EM increases its objective **monotonically**; we plot the trace to confirm convergence.

In [ ]:
def pca_emission_init(Y, dx):
    Yc = Y.reshape(-1, Y.shape[-1])
    Yc = Yc - Yc.mean(0)
    _, _, Vt = np.linalg.svd(Yc, full_matrices=False)
    return Vt[:dx].T  # (dy, dx)


def fit_lds(Y, dx, num_iters=200, seed=99, verbose=False):
    LGSSM = LinearGaussianConjugateSSM(dx, Y.shape[-1])
    params, props = LGSSM.initialize(
        jax.random.key(seed), emission_weights=jnp.array(pca_emission_init(Y, dx))
    )
    params, objective = LGSSM.fit_em(
        params, props, jnp.array(Y), num_iters=num_iters, verbose=verbose
    )
    return LGSSM, params, np.array(objective)


fit_seed = 99
m_fit, p_fit, objective = fit_lds(
    Y_train, dx, num_iters=200, seed=fit_seed, verbose=True
)
print("objective monotone non-decreasing:", bool(np.all(np.diff(objective) >= -1e-4)))

mll_fit = float(
    vmap(lambda y: m_fit.marginal_log_prob(p_fit, y))(jnp.array(Y_train)).mean()
)
print(
    f"per-sequence train marginal log-lik:  true={mll_true / n_val:.2f} (ref)   fit={mll_fit:.2f}"
)

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(objective, color="C0")
ax.set(xlabel="EM iteration", ylabel="EM objective", title="EM converges monotonically")
plt.tight_layout()
plt.show()


# The pre-EM initialization -- exactly what fit_lds started from, kept for the "how far
# did EM travel" comparisons in Sections 7 and 8. Built with the same seed as the fit.
m_init = LinearGaussianConjugateSSM(dx, Y_train.shape[-1])
p_init, _ = m_init.initialize(
    jax.random.key(fit_seed), emission_weights=jnp.array(pca_emission_init(Y_train, dx))
)


def raw(p):  # (A, C, Q, R) as numpy, in whatever basis the model happens to use
    return (
        np.array(p.dynamics.weights),
        np.array(p.emissions.weights),
        np.array(p.dynamics.cov),
        np.array(p.emissions.cov),
    )


## 5. Recovering the latent trajectory — by alignment

The fitted latents live in the model's own arbitrary basis. To compare them to the ground truth we find the linear map $T_\text{map}$ that best sends fitted smoothed means onto the true latents — this is precisely *undoing* the $\mathrm{GL}$ ambiguity, estimated by least squares. Any mismatch that survives alignment is **estimation/filtering error** (the latent is only partially determined by noisy data), *not* the basis symmetry.

In [ ]:
post = vmap(lambda y: m_fit.smoother(p_fit, y))(jnp.array(Y_train))
Xhat = np.array(post.smoothed_means)  # (n, T, dx), fitted basis

Xh = Xhat.reshape(-1, dx)
Xt = X_train.reshape(-1, dx)
W, *_ = np.linalg.lstsq(Xh, Xt, rcond=None)  # Xh @ W ~ Xt
T_map = W.T  # x_true ~ T_map . x_fit
X_aligned = (Xhat @ W).reshape(n_train, T, dx)
R2 = 1 - ((Xt - Xh @ W) ** 2).sum() / ((Xt - Xt.mean(0)) ** 2).sum()
print(f"alignment R^2 (fitted -> true latents): {R2:.3f}")

seq = 0
fig, ax = plt.subplots(dx, 1, figsize=(9, 5), sharex=True)
for i in range(dx):
    ax[i].plot(X_train[seq, :, i], "k", lw=1.6, label="true")
    ax[i].plot(X_aligned[seq, :, i], "--", color="C3", lw=1.6, label="fitted (aligned)")
    ax[i].set_ylabel(f"latent {i + 1}")
ax[0].set_title(f"Recovered latent trajectory after alignment  ($R^2={R2:.2f}$)")
ax[0].legend(frameon=False, ncol=2, loc="upper right")
ax[-1].set_xlabel("time step")
plt.tight_layout()
plt.show()

The same fit, viewed in observation space. This is the one comparison that needs **no gauge fixing**: the basis ambiguity cancels in $(CT^{-1})(Tx)=Cx$, so predicted observations are directly comparable to the data. Two bands are shown, and the gap between them is the point — the *signal* band $CV_{t\mid T}C^\top$ (how well the noise-free readout is known) is far tighter than the *predictive* band $CV_{t\mid T}C^\top + R$ (where a measured $y_t$ should land). Almost all the visible scatter of $y$ about the fit is observation noise, not estimation error. Coverage of the 95% predictive interval is a calibration check: it should come out near 0.95.

In [ ]:
# Observations reconstructed from the fitted latents. Note there is NO alignment here:
# C_fit @ x_fit lives in observation space, where the GL gauge cancels (C T^-1)(T x) = C x.
# Two bands, because there are two questions:
#   signal band     C V_{t|T} C'        -- how well do we know the noise-free readout C x_t?
#   predictive band C V_{t|T} C' + R    -- where should the *measured* y_t actually land?
C_fit = np.array(p_fit.emissions.weights)
d_fit = np.array(p_fit.emissions.bias) if p_fit.emissions.bias is not None else 0.0
R_fit = np.array(p_fit.emissions.cov)


def predict_obs(model, params, Y):
    post = vmap(lambda y: model.smoother(params, y))(jnp.array(Y))
    Xs = np.array(post.smoothed_means)  # (n, T, dx)
    Vs = np.array(post.smoothed_covariances)  # (n, T, dx, dx)
    Yh = Xs @ C_fit.T + d_fit  # (n, T, dy)
    sig_var = np.einsum("ntab,ia,ib->nti", Vs, C_fit, C_fit)
    return Yh, sig_var, sig_var + np.diag(R_fit)


Yh_tr, sig_tr, pred_tr = predict_obs(m_fit, p_fit, Y_train)
Yh_va, sig_va, pred_va = predict_obs(m_fit, p_fit, Y_val)


def obs_scores(Y, Yh, pred_var):
    r2 = 1 - ((Y - Yh) ** 2).sum() / ((Y - Y.mean((0, 1))) ** 2).sum()
    cover = np.mean(np.abs(Y - Yh) <= 2 * np.sqrt(pred_var))  # nominal 0.954
    return r2, cover


for name, Y, Yh, pv in (
    ("train", Y_train, Yh_tr, pred_tr),
    ("val  ", Y_val, Yh_va, pred_va),
):
    r2, cover = obs_scores(Y, Yh, pv)
    print(f"{name}:  R^2(y, y_hat) = {r2:.3f}   95% predictive coverage = {cover:.3f}")

seq, n_ch, t_max = 0, 5, 120
tt = np.arange(t_max)
fig, ax = plt.subplots(n_ch, 1, figsize=(9, 5.5), sharex=True)
for i in range(n_ch):
    mu_y = Yh_tr[seq, :t_max, i]
    sd_p = np.sqrt(pred_tr[seq, :t_max, i])
    sd_s = np.sqrt(sig_tr[seq, :t_max, i])
    ax[i].fill_between(
        tt,
        mu_y - 2 * sd_p,
        mu_y + 2 * sd_p,
        color="C3",
        alpha=0.15,
        label=r"$\pm2\sigma$ predictive ($C V C^\top\! + R$)",
    )
    ax[i].fill_between(
        tt,
        mu_y - 2 * sd_s,
        mu_y + 2 * sd_s,
        color="C3",
        alpha=0.40,
        label=r"$\pm2\sigma$ signal ($C V C^\top$)",
    )
    ax[i].plot(tt, Y_train[seq, :t_max, i], "k", lw=1.2, label="observed $y$")
    ax[i].plot(tt, mu_y, color="C3", lw=1.6, label=r"$C\,\hat{x}$ (smoothed)")
    ax[i].set_ylabel(f"channel {i + 1}")
r2_tr, cover_tr = obs_scores(Y_train, Yh_tr, pred_tr)
ax[0].set_title(
    f"Observations reconstructed from the fitted latents — no alignment needed "
    f"($R^2={r2_tr:.2f}$, coverage $={cover_tr:.2f}$)"
)
ax[-1].set_xlabel("time step")
handles, labels = ax[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    frameon=False,
    ncol=4,
    fontsize=9,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.055),
)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

## 6. Invariant #1 — the eigenvalues of $A$

Because an equivalent model has $A' = T A T^{-1}$, and similarity transforms preserve eigenvalues, $\operatorname{eig}(A)$ is **identifiable** even though $A$ itself is not. So the entrywise distance $\lVert A-\hat A\rVert$ can be large while the eigenvalues coincide. The cell makes the second half of that claim directly: it hits $\hat A$ with a random invertible $T$ and re-computes the spectrum, which comes back unchanged to machine precision.

In [ ]:
A_fit = np.array(p_fit.dynamics.weights)
eig_fit = np.sort_complex(np.linalg.eigvals(A_fit))
print("true eig(A):", eig_true)
print("fit  eig(A):", eig_fit)
print(
    f"\nentrywise ||A - A_fit||_F        = {np.linalg.norm(A - A_fit):.3f}   (large: different matrices)"
)
print(
    f"eigenvalue error |lam - lam_fit| = {np.abs(eig_true - eig_fit)}   (small: same spectrum)"
)

Tmap = np.random.normal(size=(dx, dx))
eig_T = np.sort_complex(np.linalg.eigvals(Tmap @ A_fit @ np.linalg.inv(Tmap)))
print(
    f"eigenvalue error after random transformation = {np.abs(eig_T - eig_fit)}   (zero: same spectrum, different basis)"
)


fig, ax = plt.subplots(figsize=(4.6, 4.6))
phi = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(phi), np.sin(phi), "k--", lw=1)
ax.scatter(
    eig_true.real,
    eig_true.imag,
    s=130,
    facecolors="none",
    edgecolors="C0",
    lw=2,
    label="true",
)
ax.scatter(eig_fit.real, eig_fit.imag, s=45, color="C3", label="fitted")
ax.axhline(0, color=".7", lw=0.6)
ax.axvline(0, color=".7", lw=0.6)
ax.set(
    xlabel="Re",
    ylabel="Im",
    title="Eigenvalues of $A$: recovered",
    aspect="equal",
    xlim=(-1.15, 1.15),
    ylim=(-1.15, 1.15),
)
ax.legend(frameon=False, loc="lower left")
plt.tight_layout()
plt.show()

## 7. Invariant #2 — the Hankel matrix and the model order

A second, deeper invariant comes from the observations' own second-order statistics — and unlike the eigenvalues of $A$, you can build it without a model at all.

**Start from the data.** For a stationary process, the lag-$k$ output covariance
$$
\Lambda_k \;=\; \mathbb E\!\left[y_{t+k}\,y_t^\top\right] \qquad (k\ge 1)
$$
is something you can estimate from any recording in three lines. Stack the lags into a block-**Hankel** matrix whose $(i,j)$ block is $\Lambda_{i+j-1}$:
$$
\mathcal H_m=\begin{bmatrix}\Lambda_1&\Lambda_2&\cdots&\Lambda_m\\ \Lambda_2&\Lambda_3&&\vdots\\ \vdots&&\ddots&\\ \Lambda_m&\cdots&&\Lambda_{2m-1}\end{bmatrix}.
$$

**The model factors it.** Let $\Pi$ solve the discrete Lyapunov equation $\Pi = A\Pi A^\top + Q$ (the stationary latent covariance) and set $G = A\Pi C^\top$. Then $\Lambda_k = C A^{\,k-1} G$, so each block splits,
$$
\Lambda_{i+j-1} \;=\; C A^{\,i+j-2} G \;=\; \underbrace{\big(C A^{\,i-1}\big)}_{\text{depends only on }i}\;\underbrace{\big(A^{\,j-1} G\big)}_{\text{depends only on }j},
$$
and therefore so does the whole matrix — into a **tall** factor times a **wide** one:
$$
\mathcal H_m=\underbrace{\begin{bmatrix}C\\ CA\\ \vdots\\ CA^{m-1}\end{bmatrix}}_{\textbf{future (query) matrix}\ \mathcal O_m}\;
\underbrace{\big[\,G\ \ AG\ \ \cdots\ \ A^{m-1}G\,\big]}_{\textbf{past (key) matrix}\ \mathcal C_m}.
$$
$\mathcal O_m$ describes how the current state is broadcast into the next $m$ observations — the **future**. $\mathcal C_m$ describes how the previous $m$ observations deposit into the current state — the **past**. *(These are the* observability *and* controllability *matrices of the systems literature. There is no controller and no input anywhere in this model, so we use the plainer names.)*

Each factor has only $d_x$ columns or rows, so $\operatorname{rank}\mathcal H_m = d_x$: **the number of non-zero Hankel singular values is the state dimension.** Those singular values are basis-invariant (they are functions of $y$ alone), so the true model, the fitted model, and even the *raw sample covariances* should share the same rank-$d_x$ fingerprint. This is the seed of subspace identification (Ho–Kalman / N4SID).

**And there is the gauge, in one line.** The factorization is not unique. For any invertible $T$,
$$
\mathcal H_m \;=\; \big(\mathcal O_m T^{-1}\big)\big(T\,\mathcal C_m\big)
$$
is an equally good split into future times past. **The product is what the data pin down; the factors are what EM returns.** That is the identifiability problem of Section 1, now visible as a property of a single matrix — and Section 8 resolves it by choosing the split canonically.

Because $\mathcal H_m$ is assembled purely from output covariances it is **gauge-free** — no alignment enters anywhere — and for minimal models it is *complete*: two models produce the same $\mathcal H_m$ if and only if they are the same system up to a change of latent basis. So $\lVert\mathcal H_m - \hat{\mathcal H}_m\rVert_F / \lVert\mathcal H_m\rVert_F$ is the whole-system counterpart of the per-matrix errors in Section 8, and unlike those it makes a single joint statement about $(A, C, Q)$. We report it for the fitted model, the pre-EM initialization, and the raw sample covariances.

In [ ]:
def hankel_from_params(A, C, Q, R, m=10):
    Pi = solve_discrete_lyapunov(A, Q)
    G = A @ Pi @ C.T
    Lam = [
        C @ np.linalg.matrix_power(A, k - 1) @ G for k in range(1, 2 * m)
    ]  # Lam_1..Lam_{2m-1}
    d = C.shape[0]
    H = np.zeros((d * m, d * m))
    for i in range(m):
        for j in range(m):
            H[i * d : (i + 1) * d, j * d : (j + 1) * d] = Lam[i + j]
    return H


def hankel_from_data(Y, m=10):
    Yc = Y - Y.mean(axis=(0, 1))
    Tt = Yc.shape[1]
    d = Yc.shape[-1]

    def lam(k):
        return np.einsum("nti,ntj->ij", Yc[:, k:, :], Yc[:, : Tt - k, :]) / (
            Yc.shape[0] * (Tt - k)
        )

    Lam = [lam(k) for k in range(1, 2 * m)]
    H = np.zeros((d * m, d * m))
    for i in range(m):
        for j in range(m):
            H[i * d : (i + 1) * d, j * d : (j + 1) * d] = Lam[i + j]
    return H


xcov_lag = 10
H_true = hankel_from_params(A, C, Q, R, m=xcov_lag)
H_fit = hankel_from_params(*raw(p_fit), m=xcov_lag)
H_init = hankel_from_params(*raw(p_init), m=xcov_lag)
H_emp = hankel_from_data(Y_train, m=xcov_lag)
svals = lambda H: np.linalg.svd(H, compute_uv=False)
sv_true, sv_fit, sv_emp = svals(H_true), svals(H_fit), svals(H_emp)

# The Hankel matrix is built only from output covariances, so it is gauge-free: no
# alignment is involved, and two minimal models share it iff they are the same system
# up to a change of latent basis.
print("relative Frobenius distance to the true Hankel matrix (no alignment needed):")
for nm, H in (
    ("fitted model", H_fit),
    ("initial model", H_init),
    ("raw sample covariances", H_emp),
):
    print(f"   {nm:24s}{np.linalg.norm(H - H_true) / np.linalg.norm(H_true):8.3f}")
print()
k = np.arange(1, xcov_lag + 1)
fig, ax = plt.subplots(figsize=(7, 3.8))
ax.semilogy(k, sv_true[:xcov_lag] / sv_true[0], "o-", label="true model")
ax.semilogy(k, sv_fit[:xcov_lag] / sv_fit[0], "s--", label="fitted model")
ax.semilogy(
    k, sv_emp[:xcov_lag] / sv_emp[0], "^:", color="C2", label="raw sample covariances"
)
ax.axvline(dx, color="C3", lw=1.2, ls=":")
ax.text(dx + 0.1, 3e-1, f"$d_x={dx}$", color="C3")
ax.set(
    xlabel="singular value index",
    ylabel=r"$ V_i/ V_1$",
    title="Hankel singular values: a cliff at the true state dimension",
    ylim=(1e-3, 2),
)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()
print("normalized sv (true model):", np.round(sv_true[:6] / sv_true[0], 4))

## 8. Parameter recovery: comparing every matrix, in a canonical basis

Section 5 removed the gauge by *estimating* it, regressing fitted latents on true ones to get $\hat T$. That is fine for looking at trajectories, but any comparison built on it inherits whatever error is in $\hat T$. Here we instead put each model into a canonical basis on its own, then compare the results.

**Two Gram matrices.** Collapse the Section 7 factors onto the latent space, the tall future one from the left and the wide past one from the right:
$$
\Omega_{\rightarrow} \;=\; \mathcal O_m^\top \mathcal O_m \;=\; \sum_{k}(A^\top)^k\,C^\top C\,A^k,
\qquad
\Omega_{\leftarrow} \;=\; \mathcal C_m \mathcal C_m^\top \;=\; \sum_{k}A^k\,GG^\top\,(A^\top)^k .
$$
Both are $d_x\times d_x$. $u^\top\Omega_{\rightarrow}u$ says how loudly latent direction $u$ shows up in the **future** of the data, $u^\top\Omega_{\leftarrow}u$ how loudly it shows up in the **past**. *(These are the* observability *and* controllability *Gramians of the systems literature.)*

**Why an invariant exists.** The gauge acts on the two factors oppositely, $\mathcal O_m \mapsto \mathcal O_m T^{-1}$ and $\mathcal C_m \mapsto T\,\mathcal C_m$, so
$$
\Omega_{\rightarrow}\mapsto T^{-\top}\Omega_{\rightarrow}T^{-1},
\qquad
\Omega_{\leftarrow}\mapsto T\,\Omega_{\leftarrow}T^{\top},
\qquad
\Omega_{\leftarrow}\Omega_{\rightarrow}\;\mapsto\;T\big(\Omega_{\leftarrow}\Omega_{\rightarrow}\big)T^{-1}.
$$
The product is only conjugated, and eigenvalues survive conjugation, so $\sqrt{\operatorname{eig}(\Omega_{\leftarrow}\Omega_{\rightarrow})}$ is gauge-free: these are the Hankel singular values $\sigma_1\ge\cdots\ge\sigma_{d_x}$ of Section 7.

**Balancing.** Take the SVD $\mathcal H_m = U\Sigma V^\top$. Every valid split into future times past has the form $\mathcal O_m = U\Sigma^{1/2}T^{-1}$, $\mathcal C_m = T\,\Sigma^{1/2}V^\top$, which makes the gauge concrete: **it is the freedom in how you divide $\Sigma$ between past and future**, and EM returns an arbitrary division. The **balanced** realization takes $T=I$, giving each side $\sqrt\Sigma$, so that $\Omega_{\rightarrow}=\Omega_{\leftarrow}=\Sigma$. Two models related by a change of basis share $\mathcal H_m$, hence share $U\Sigma V^\top$, hence land on identical balanced parameters. The construction uses one model at a time, so nothing inherits error from a fitted $\hat T$. Each matrix then gets one number,
$$
\text{rel. error} \;=\; \lVert \hat P - P\rVert_F \,/\, \lVert P\rVert_F,
$$
zero exactly when the two matrices agree. $R$ never sees the latent basis and needs no transformation.

**The residual freedom.** Balancing does not quite pin down $T$. If $T$ and $\tilde T$ both balance the same model, $S=\tilde T T^{-1}$ satisfies both $S\Sigma S^\top=\Sigma$ and $S^\top\Sigma S=\Sigma$, which forces $S$ orthogonal and commuting with $\Sigma$: block diagonal, one block per *repeated* singular value. Distinct sorted $\sigma$ leave only $S=\operatorname{diag}(\pm1)$, sign flips and no permutations; a $\sigma$ of multiplicity $k$ leaves a full $O(k)$. Oscillatory modes are the awkward case, since a rotational pair produces two nearly equal $\sigma$. We therefore group $\sigma$'s within a relative tolerance and fit an orthogonal Procrustes on the balanced loadings inside each group, which for a block of size 1 is just a sign flip. With only *nearly* equal $\sigma$ the exact group is still $\operatorname{diag}(\pm1)$, so allowing $O(k)$ is a deliberate slackening to absorb the ill-conditioning of the eigenvector step, not an extra symmetry of the model. The cell prints the block sizes and the smallest relative $\sigma$ gap; conditioning scales like 1/gap.

**Two implementation notes.** We weight $\Omega_{\rightarrow}$ with $C^\top C$ rather than $C^\top R^{-1}C$. The latter is the Fisher information the observations carry about the state and orders directions by estimability rather than raw output energy, but it makes the basis depend on $\hat R$ and lets error there leak into the $A,C,Q$ comparison. And the Grams are never built from stacks (at $d_y=50$, $m=10$ each is $500\times d_x$); peeling off the $k=0$ term makes each one the fixed point of a discrete Lyapunov equation,
$$
\Omega_{\rightarrow} = A^\top\Omega_{\rightarrow}A + C^\top C,
\qquad
\Omega_{\leftarrow} = A\,\Omega_{\leftarrow}A^\top + GG^\top,
$$
one `solve_discrete_lyapunov` call each. This is the $m\to\infty$ limit, so the $\sigma_i$ printed here are the asymptotic values that Section 7's finite-lag ones converge to.

**What to watch.** Balancing spends all $d_x^2$ gauge parameters, so the per-matrix errors are not independent of each other, and the whole construction is well conditioned only when the $\sigma_i$ are well separated. Two self-checks print alongside the table: applying a random $T$ to the truth and re-balancing should return the same matrices (gauge invariance), and balancing an already-balanced model should change nothing (idempotence). Both should land at machine precision; if they do not, the residual freedom was resolved wrongly and every number in the table is suspect. We repeat the comparison for the pre-EM initialization to show how far EM travels. *(Heat maps share one color scale per row, so a scale error is visible rather than normalized away.)*

In [ ]:
def canonical(A_, C_, Q_, R_):
    # BALANCED realization -- the canonical gauge. Section 7 split the Hankel matrix into a
    # tall future factor O_m and a wide past factor C_m; collapse each onto the latent space
    # (the tall one from the left, the wide one from the right) to get two d_x by d_x Grams:
    #   Gram_future = O_m' O_m = sum_k (A')^k C'C A^k     how loudly a latent direction
    #   Gram_past   = C_m C_m' = sum_k A^k GG' (A')^k     shows up in the future / the past
    # Balancing splits the Hankel SVD down the middle, handing each side sqrt(Sigma), which
    # makes both Grams equal to Sigma = diag(Hankel singular values). It is computed from one
    # model alone -- no reference, no estimated alignment -- so nothing here inherits error
    # from a fitted T. The Lyapunov solves below are the m -> inf limit of those two sums.
    # Weighting note: C'C measures raw output energy. C' R^-1 C is equally valid and is the
    # Fisher information about the state, which orders directions by estimability rather
    # than by energy -- but it makes the basis depend on Rhat, so Rhat's error would leak
    # into the A / C / Q comparison. We keep C'C so the basis depends only on (A, C, Q).
    Pi = solve_discrete_lyapunov(A_, Q_)  # stationary latent covariance
    G = A_ @ Pi @ C_.T  # lag-1 cross-covariance
    Gram_past = solve_discrete_lyapunov(A_, G @ G.T)
    Gram_future = solve_discrete_lyapunov(A_.T, C_.T @ C_)
    L = np.linalg.cholesky((Gram_past + Gram_past.T) * 0.5)
    w, U = np.linalg.eigh(L.T @ ((Gram_future + Gram_future.T) * 0.5) @ L)
    idx = np.argsort(w)[::-1]
    hsv = np.sqrt(np.clip(w[idx], 1e-12, None))  # Hankel singular values, descending
    U = U[:, idx]
    Tinv = L @ U @ np.diag(hsv**-0.5)
    T = np.diag(hsv**0.5) @ U.T @ solve_triangular(L, np.eye(dx), lower=True)
    assert np.allclose(T @ Tinv, np.eye(dx), atol=1e-6), (
        "balancing transform inconsistent"
    )
    return dict(A=T @ A_ @ Tinv, C=C_ @ Tinv, Q=T @ Q_ @ T.T, R=R_), hsv


def resolve(P_ref, P, hsv, tol=0.05):
    # Resolve the residual freedom of the balanced form against a reference.
    # If T and T~ both balance the same model, S = T~ T^-1 obeys S Sig S' = Sig AND
    # S' Sig S = Sig, which forces S orthogonal and commuting with Sig: block diagonal,
    # one block per REPEATED singular value. With the sigma sorted and distinct the group
    # is therefore just diag(+-1) -- sign flips, no permutations. We treat sigma within a
    # relative distance `tol` as one block and allow a full O(k) rotation inside it. With
    # only *nearly* equal sigma the exact group is still diag(+-1), so this is a deliberate
    # slackening that absorbs the ill-conditioning of the eigenvector step, not an extra
    # symmetry of the model. Within each block we take the orthogonal Procrustes fit of the
    # balanced loadings; for a 1x1 block that reduces exactly to a sign flip.
    cuts = (
        [0]
        + [i for i in range(1, dx) if (hsv[i - 1] - hsv[i]) / hsv[i - 1] > tol]
        + [dx]
    )
    S = np.zeros((dx, dx))
    for lo, hi in zip(cuts[:-1], cuts[1:]):
        b = slice(lo, hi)
        Uo, _, Vto = np.linalg.svd(P["C"][:, b].T @ P_ref["C"][:, b])
        S[b, b] = Uo @ Vto
    out = dict(A=S.T @ P["A"] @ S, C=P["C"] @ S, Q=S.T @ P["Q"] @ S, R=P["R"])
    return out, [hi - lo for lo, hi in zip(cuts[:-1], cuts[1:])]


P_true, hsv_true = canonical(A, C, Q, R)
P_fit, hsv_fit = canonical(*raw(p_fit))
P_init, hsv_init = canonical(*raw(p_init))
P_fit, blocks_fit = resolve(P_true, P_fit, hsv_true)
P_init, _ = resolve(P_true, P_init, hsv_true)


def rel_fro(
    P, Phat
):  # zero iff the matrices agree, given the residual is resolved right
    return float(np.linalg.norm(Phat - P) / np.linalg.norm(P))


names = ["A", "C", "Q", "R"]
lab = {
    "A": "dynamics $A$",
    "C": "loadings $C$",
    "Q": "process cov $Q$",
    "R": "obs. noise cov $R$",
}

print("relative Frobenius error in the balanced (canonical) basis")
print(f"{'parameter':12s}{'fitted':>10s}{'initial':>10s}")
for nm in names:
    print(
        f"{nm:12s}{rel_fro(P_true[nm], P_fit[nm]):>10.3f}"
        f"{rel_fro(P_true[nm], P_init[nm]):>10.3f}"
    )

# ---- does the canonical form do what it claims? ----
gap = (hsv_true[:-1] - hsv_true[1:]) / hsv_true[:-1]
print(f"\nHankel singular values (m -> inf), true : {np.round(hsv_true, 3)}")
print(f"                                 fitted : {np.round(hsv_fit, 3)}")
print(
    f"smallest relative gap sigma_i -> sigma_i+1 = {gap.min():.3f}"
    f"   (eigenvector conditioning scales like 1/gap)"
)
print(
    f"residual-freedom blocks resolved: {blocks_fit}  (a block of size k > 1 means an"
    f" O(k) rotation was fitted, not just a sign)"
)

rng_chk = np.random.default_rng(0)
Tc = rng_chk.normal(size=(dx, dx))
Tci = np.linalg.inv(Tc)
P_g, hsv_g = canonical(Tc @ A @ Tci, C @ Tci, Tc @ Q @ Tc.T, R)  # same model, new gauge
P_g, _ = resolve(P_true, P_g, hsv_true)
P_i2, hsv_i2 = canonical(*[P_true[n] for n in names])  # balance the balanced form
P_i2, _ = resolve(
    P_true, P_i2, hsv_true
)  # a fixed point only modulo the residual group
print(
    "gauge invariance : random T applied to the truth, re-balanced -> max |diff| = "
    f"{max(np.abs(P_g[n] - P_true[n]).max() for n in names):.2e}"
    f"   |sigma diff| = {np.abs(hsv_g - hsv_true).max():.2e}"
)
print(
    "idempotence      : balancing the balanced form (mod residual group) -> max |diff| = "
    f"{max(np.abs(P_i2[n] - P_true[n]).max() for n in names):.2e}"
    f"   |sigma diff| = {np.abs(hsv_i2 - hsv_true).max():.2e}"
)

# ---- triplet heat maps:  generating | fitted | initial ----
# One color scale per row (set by the generating matrix), so a scale error is visible
# rather than normalized away -- the balanced basis makes the panels directly comparable.
fig, ax = plt.subplots(len(names), 3, figsize=(7.5, 2.5 * len(names)))
for r, nm in enumerate(names):
    v = np.abs(P_true[nm]).max()
    trip = [
        ("generating", P_true[nm]),
        (f"fitted  rel.err$={rel_fro(P_true[nm], P_fit[nm]):.3f}$", P_fit[nm]),
        (f"initial  rel.err$={rel_fro(P_true[nm], P_init[nm]):.3f}$", P_init[nm]),
    ]
    for c, (ttl, Pm) in enumerate(trip):
        a = ax[r, c]
        a.imshow(Pm, cmap="RdBu_r", vmin=-v, vmax=v, aspect="auto")
        a.set_xticks([])
        a.set_yticks([])
        a.set_title(ttl, fontsize=9)
        if c == 0:
            a.set_ylabel(lab[nm], fontsize=11)
fig.suptitle("Parameter recovery in the balanced basis (shared scale per row)", y=1.0)
plt.tight_layout()
plt.show()

## 9. Model selection by cross-validated log-likelihood

In practice $d_x$ is unknown. We fit models of increasing latent dimension and score each by its **held-out predictive log-likelihood** — the marginal log-likelihood of the *validation* sequences under the fitted parameters, per time step. We prefer this to AIC/BIC: it measures generalization directly and needs no parameter-counting penalty (which is awkward here anyway, given the flat likelihood directions). The curve should rise until $d_x^\star$ and then flatten — extra latent dimensions should not improve prediction of data generated by a $d_x$-dimensional process.

In [ ]:
dims = range(1, 11)
val_ll = []
for k in dims:
    mk, pk, _ = fit_lds(Y_train, k, num_iters=200, verbose=False)
    ll = float(vmap(lambda y: mk.marginal_log_prob(pk, y))(jnp.array(Y_val)).sum()) / (
        n_val * T
    )
    val_ll.append(ll)
    print(f"  d_x={k}:  val log-lik/step = {ll:.4f}")
best = list(dims)[int(np.argmax(val_ll))]

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(list(dims), val_ll, "o-", color="C0")
ax.axvline(best, color="C3", ls=":")
ax.axvline(dx, color="k", ls="--", alpha=0.4)
ax.set(
    xlabel="latent dimension $d_x$",
    ylabel="validation log-lik / step",
    title=f"Cross-validated model selection picks $d_x={best}$",
)
plt.tight_layout()
plt.show()

## Recap & what's next

- A linear–Gaussian SSM is identifiable **only up to an invertible change of latent basis** ($x_t\mapsto Tx_t$). The likelihood is flat along that orbit; raw parameters and latent coordinates are not estimable.
- **What is estimable:** invariants of the orbit — the **eigenvalues of $A$**, the **Hankel singular values** (which also reveal the **model order** $d_x$) — the latent trajectory *after alignment*, and the observations themselves, which need no gauge fixing at all because $(CT^{-1})(Tx)=Cx$.
- **The full parameter set is comparable too**, but only after each model is independently put into the **balanced canonical realization**: no estimated $\hat T$, nothing that inherits error from an alignment, and one relative-Frobenius number per matrix that is zero iff the matrices agree.
- EM (here MAP-EM for stability) recovers the model up to that symmetry; **cross-validated log-likelihood** selects $d_x$, agreeing with the Hankel cliff.

**Next.** So far the dynamics $A$ were fixed for all time. **Notebook 3** lets the system *switch* between several linear regimes — a **switching LDS** — and fits it to resting-state fMRI, asking whether the brain's spontaneous activity is better described by one linear process or several, and what dynamical "modes" those regimes correspond to.